In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# normal data inspection imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/cleaned_reviews.csv')

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming tech...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically theres a family where a little boy j...,0
4,petter matteis love in the time of money is a ...,1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorize = TfidfVectorizer(max_features=2000)

In [ ]:
X = vectorize.fit_transform(df['review'])
y = df['sentiment']

In [ ]:
type(X)

scipy.sparse._csr.csr_matrix

In [ ]:
print(X[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 121 stored elements and shape (1, 2000)>
  Coords	Values
  (0, 1221)	0.027304403926657703
  (0, 1207)	0.12599037935690077
  (0, 1731)	0.25872431365698745
  (0, 1238)	0.07982049844455329
  (0, 1439)	0.09680989952389463
  (0, 771)	0.03285642474724001
  (0, 1093)	0.0844815623894282
  (0, 1729)	0.08400281845625399
  (0, 38)	0.04292314623909334
  (0, 1898)	0.09926261296984111
  (0, 928)	0.0641842793794452
  (0, 537)	0.1469346052138817
  (0, 1989)	0.06994146986578575
  (0, 149)	0.053955211273541206
  (0, 1744)	0.03231310912832955
  (0, 100)	0.05453060583166028
  (0, 1445)	0.11115883951948763
  (0, 110)	0.09948569760692816
  (0, 1753)	0.056775481167952356
  (0, 894)	0.1710920905763869
  (0, 557)	0.07387956464340814
  (0, 1914)	0.06776653993838792
  (0, 764)	0.07396513019573601
  (0, 1944)	0.11678364532525944
  (0, 648)	0.08244167053885763
  :	:
  (0, 739)	0.0537736821545319
  (0, 1242)	0.032120119876212896
  (0, 939)	0.072226656922

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.88      0.88      5000
           1       0.88      0.89      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [ ]:
new_sentence ="this movie is flop"
vector = vectorize.transform([new_sentence])
prediction = model.predict(vector)
print(prediction)

[0]


In [ ]:
new_sentence ="this movie is great"
vector = vectorize.transform([new_sentence])
prediction = model.predict(vector)
print(prediction)

[1]


In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])
pipeline.fit(X_train, y_train)

joblib.dump(pipeline, 'sentiment_model.joblib')

['sentiment_model.joblib']

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV

In [ ]:
X_raw = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,       # limit vocabulary for speed and memory
    ngram_range=(1,2),       # unigrams and bigrams often help
    stop_words='english'     # remove common words
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_tfidf, y_train)
y_pred_dt = dt.predict(X_test_tfidf)

print("Decision Tree – Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree – Accuracy: 0.7236
              precision    recall  f1-score   support

           0       0.72      0.73      0.73      5000
           1       0.73      0.72      0.72      5000

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72     10000
weighted avg       0.72      0.72      0.72     10000



In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', DecisionTreeClassifier(random_state=42))
])
pipeline.fit(X_train, y_train)

joblib.dump(pipeline, 'decision_tree_model.joblib')

['decision_tree_model.joblib']

In [ ]:
pipeline_rf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline_rf.fit(X_train, y_train)
joblib.dump(pipeline_rf, 'random_forest_model.joblib')

['random_forest_model.joblib']